In [ ]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [5]:
load_dotenv(override=True)

True

In [6]:
class ConfigureModel:

    def __init__(self, model_name:str="openai/gpt-oss-20b", groq_api_key:str=os.getenv('GROQ_API_KEY')):
        self.model_name = model_name
        self.groq_api_key = groq_api_key
        self.model = None
        self._initiate_model()
    
    def _initiate_model(self):
        print(f"Trying to connect with model {self.model_name}...")
        try:
            self.model = ChatGroq(
                model = self.model_name,
                temperature = 0.7,
                groq_api_key = self.groq_api_key
            )
            print("Model connection successfull")
        
        except Exception as e:
            print(f"Error: {e}")

    def get_response(self, prompt):
        print(f"prompt sent to model, waiting for response")
        try:
            response = self.model.invoke(prompt)
            return response.content
        
        except Exception as e:
            print(f"Error: {e}")        

In [7]:
model_configurer = ConfigureModel()

Trying to connect with model openai/gpt-oss-20b...
Model connection successfull


In [15]:
prompt1 = PromptTemplate(
    template = "give top 5 keywords about the topic {topic}",
    input_variables = ['topic']
)
prompt2 = PromptTemplate(
    template = "give 2 line summary of the topic {topic}",
    input_variables = ['topic']
)
prompt3= PromptTemplate(
    template = "combine given results and give a final result. result1--> {result1}, result2--> {result2}",
    input_variables = ['result1', 'result2']
)

In [13]:
parser = StrOutputParser()

## Creating Parallel Chains

In [21]:
parallel_chains = RunnableParallel({
    'result1' : prompt1 | model_configurer.model | parser,
    'result2' : prompt2 | model_configurer.model | parser
})

In [22]:
chain3 = prompt3 | model_configurer.model | parser

### Making final chain

In [23]:
# here first two parallel chains would be executed and then chain3 would combine their results, this as whole is chain now
final_chain = parallel_chains | chain3

In [24]:
result = final_chain.invoke({'topic' : 'Machine Learning'})

In [25]:
print(result)

**Machine Learning Overview**

Machine Learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance on tasks without explicit programming. By extracting patterns and building predictive models, it powers applications from image recognition and natural language processing to autonomous systems and personalized recommendations.

**Top 5 Keywords for Machine Learning**

1. **Supervised Learning** – learning from labeled data to make predictions.  
2. **Neural Networks** – layered models that mimic the brain’s structure for complex pattern recognition.  
3. **Training Data** – the dataset used to teach models and validate their accuracy.  
4. **Model Evaluation** – techniques (cross‑validation, metrics, etc.) to assess performance and generalization.  
5. **Feature Engineering** – the process of selecting, transforming, and creating input variables that enhance model effectiveness.


In [26]:
final_chain.get_graph().print_ascii()

        +--------------------------------+         
        | Parallel<result1,result2>Input |         
        +--------------------------------+         
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
    +----------+                  +----------+     
    | ChatGroq |                  | ChatGroq |     
    +----------+                  +----------+     
          *                             *          
          *                             *          
          *                             *          
+-----------------+            +-----------------+ 
| StrOutputP